# Simple EDA to run on the dataset generated by the task generated dataset

# Read in, Generate features

In [ ]:
# installs
# %pip install hipe4ml uproot pandas numpy matplotlib seaborn scikit-learn


In [ ]:
import hipe4ml as h4ml
import uproot
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from hipe4ml.tree_handler import TreeHandler
import pandas as pd
import Utils
import importlib
from scipy.stats import ks_2samp

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df_original = Utils.get_dataframe("FwdMatchMLCandidatesFull.root")

In [ ]:
df =df_original.copy()
importlib.reload(Utils)

In [ ]:
df.describe()

In [ ]:
df = Utils.process_dataframe(df, makedummies=True)
features=df.columns.tolist() # add the dummmy features to the list of features

In [ ]:
df[features].describe()

# Feature Plots

In [ ]:
# Disabled for performance + match type breakdown plots are more informative
# def hist_trimmed(df, col, per=0.001, bins=100, log=False):
#     low = df[col].quantile(per)
#     high = df[col].quantile(1 - per)
    
#     data = df.loc[(df[col] >= low) & (df[col] <= high), col]
    
#     sns.histplot(data, bins=bins, log_scale=log)
# # Disabled due to performance
# for entry in features:
#     hist_trimmed(df,entry,per=0.005)
#     plt.show()

# Feature Plots match type breakdown

In [ ]:

# Reworked to commonalities
match_groups = Utils.build_match_groups(df)
Utils.draw_all_features(features, match_groups, per=0.005, density=True)

In [ ]:
## SKIPPED DUE TO DEMANDING NATURE
# plt.figure(figsize=(100, 80))
# corr_matrix = df[features].corr()
# sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,     fmt=".2f")         # <-- format to 2 decimal places)
# plt.title('Correlation Heatmap')
# plt.show()

In [ ]:
## SKIPPED DUE TO DEMANDING NATURE

# #Pairplot to get an idea of variable dependencies
# pp = sns.pairplot(df[features].sample(n=1000), hue="IsSignal", plot_kws={'alpha': 0.25})
# plt.suptitle('Pair Plot of Selected Features', y=1.02)

# for ax in pp.axes.flat:
#     ax.tick_params(axis='both', labelleft=True, labelbottom=True)
    
# plt.subplots_adjust(wspace=0.3, hspace=0.3)
# plt.show()

# Multiplicity and Miscellaneous

In [ ]:
# Compute the number of entries per mchID
multiplicities = df['mchID'].value_counts()

# Compute the frequency of each multiplicity
multiplicity_counts = multiplicities.value_counts().sort_index()

# Plot the distribution
plt.figure(figsize=(10,6))
plt.bar(multiplicity_counts.index, multiplicity_counts.values)
plt.xlabel('Multiplicity (number of entries per mchID)')
plt.ylabel('Frequency')
plt.title('Multiplicity Distribution of mchID')
plt.show()

In [ ]:
# Group by mchID and check if any IsSignal == 1
group_has_match = df.groupby('mchID')['IsSignal'].max()  # max will be 1 if any is 1

# Count how many groups have match (1) and no match (0)
has_match_count = (group_has_match == 1).sum()
no_match_count = (group_has_match == 0).sum()

# Pie chart
labels = ['Has Match', 'No Match']
sizes = [has_match_count, no_match_count]
plt.figure(figsize=(8,8))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140)
plt.title('Percentage of mchID Groups with at Least One Match (IsSignal=1)')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

# Extrapolate to the 8 match labels fake, decay etc.
# IsSignal is only true matches 1 is true only for true matching (leading+nonleading)

In [ ]:

def plot_match_label_breakdown(df: pd.DataFrame,
                               match_label_groups: dict,
                               title: str = "Match Label Breakdown") -> None:
    """
    Plots grouped bar chart using MATCH_LABEL_GROUPS.

    Automatically handles:
    - Leading vs non-leading (if 2 codes exist)
    - Single-code groups (e.g. Dummy match)
    """

    categories = list(match_label_groups.keys())
    total = len(df)

    leading_fracs = []
    nonleading_fracs = []

    for cat in categories:
        codes = match_label_groups[cat]

        # Leading = first code
        lead_code = codes[0]
        lead_frac = df["MatchLabel"].eq(lead_code).sum() / total

        # Non-leading = second code if exists
        if len(codes) > 1:
            nonlead_code = codes[1]
            nonlead_frac = df["MatchLabel"].eq(nonlead_code).sum() / total
        else:
            nonlead_frac = 0.0  # e.g. Dummy match

        leading_fracs.append(lead_frac)
        nonleading_fracs.append(nonlead_frac)

    x = np.arange(len(categories))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))

    bars_l = ax.bar(x - width/2, leading_fracs, width,
                    label="Leading", alpha=0.85)

    bars_n = ax.bar(x + width/2, nonleading_fracs, width,
                    label="Non-leading", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(categories, rotation=20)
    ax.set_ylabel("Fraction of total candidates")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    # Annotate bars
    for bar in [*bars_l, *bars_n]:
        height = bar.get_height()
        if height > 0.001:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.002,
                f"{height:.3f}",
                ha="center", va="bottom", fontsize=8
            )

    plt.tight_layout()
    plt.show()


plot_match_label_breakdown(df,match_label_groups=Utils.MATCH_LABEL_GROUPS, title="Match Label Breakdown")

# KS breakdown

In [ ]:
def compute_ks(df, feature, label_col="IsSignal"):
    sig = df[df[label_col] == 1][feature]
    bkg = df[df[label_col] == 0][feature]

    ks_stat, p_value = ks_2samp(sig, bkg)
    
    return ks_stat, p_value

def ks_scan(df, features, label_col="IsSignal"):
    results = []

    for feat in features:
        ks, p = compute_ks(df, feat, label_col)
        results.append({
            "feature": feat,
            "ks_stat": ks,
            "p_value": p
        })

    return pd.DataFrame(results).sort_values("ks_stat", ascending=False)

def plot_ks_ranking(df_ks, height_per_feature=0.4, min_height=6):
    """
    Plot KS ranking with automatic vertical scaling.
    
    Parameters
    ----------
    df_ks : pd.DataFrame
        Must contain columns ['feature', 'ks_stat']
    height_per_feature : float
        Height (in inches) per feature
    min_height : float
        Minimum figure height
    """

    df_ks = df_ks.sort_values("ks_stat")

    n_features = len(df_ks)
    fig_height = max(min_height, n_features * height_per_feature)

    plt.figure(figsize=(10, fig_height))

    plt.barh(df_ks["feature"], df_ks["ks_stat"])

    plt.xlabel("KS statistic")
    plt.ylabel("Feature")
    plt.title("Feature separation power (KS)")

    plt.grid(True, axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()

df_ks = ks_scan(df, features)
print(df_ks)
plot_ks_ranking(df_ks)

In [ ]:

df['DeltaDirection'].max()